In [142]:
import pandas as pd
import openpyxl
import numpy as np
from __future__ import unicode_literals
from collections import defaultdict

from lxml import etree
from ranking import Ranking
import requests

from trueskill import Rating, rate_1vs1, quality_1vs1, rate, quality
from trueskill.mathematics import Gaussian

df = pd.read_excel(
        io='MensRaceResults.xlsm',
        engine ='openpyxl',
        sheet_name='Sprint',
        skiprows=0,
        usecols='A:S',
        nrows=3000
        )

In [143]:
df = df.replace({' ': '_'}, regex=True)
df = df.replace({'’': "'"}, regex=True)
df.insert(4, "Round", 1, True)
df.insert(15, "Avg Speed", 1, True)
df["Avg Speed"] = df["Avg Speed R1"]

df_2 = df.loc[(df['Rank R2'] >0) ]
#df_2.insert(10, "Round", 2, True)
df_3 = df.loc[(df['Rank R3'] >0) ]
#df_3.insert(10, "Round", 3, True)

df_2["Round"] = 2
df_2["Rank R1"]=df_2["Rank R2"]
df_2["200m R1"]=df_2["200m R2"]
df_2["Avg Speed"] = df_2["Avg Speed R2"]

df_3["Round"] = 3
df_3["Rank R1"]=df_3["Rank R3"]
df_3["200m R1"]=df_3["200m R3"]
df_3["Avg Speed"] = df_3["Avg Speed R3"]

df_final = pd.concat([df,df_2,df_3])

#df_final["Avg Speed"] = df_final["Avg Speed R1"]
df_final.insert(11, "Rank", 1, True)
df_final.insert(12, "Time", 1, True)
df_final["Rank"] = df_final["Rank R1"]
df_final["Time"] = df_final["200m R1"]



df_final = df_final.drop(columns=['Rank R1', '200m R1', 'Avg Speed R1','Rank R2', '200m R2', 'Avg Speed R2','Rank R3', '200m R3', 'Avg Speed R3'])



df = df_final
df = df.reset_index()
df.insert(15,"Initial_CSE", "Holder", True)
df.insert(16,"Final_CSE", "Holder", True)
df.insert(17,"Mu", "Holder", True)
df.insert(18,"Sigma", "Holder", True)
df.insert(19,"ExpectedRank", "Holder", True)
df.insert(20,"RatingChange", "Holder", True)


df['Stage'] = pd.Categorical(df['Stage'], ["R32", "Rep32", "R16","Rep16","R8","Rep8","QF","5to8","SF","F"])

df=df.sort_values(["Date","Stage"])
df = df.reset_index()
df=df.drop(columns=["level_0","index"])
new_row = {'Location':'TEMP ROW', 'Year':3000, 'Round':1, 'Heat':1}
df = df.append(new_row, ignore_index=True)

In [144]:
df

,Location,Year,Date,Event,Round,Stage,Heat,Athlete,Country,Age,Final_Rank,Rank,Time,Avg Speed,Initial_CSE,Final_CSE,Mu,Sigma,ExpectedRank,RatingChange
0,Milton,2018,2018-10-28,NC,1,R16,1,BARRETTE_Hugo,CAN,27.336986,16.0,1.0,10.178,70.740814,Holder,Holder,Holder,Holder,Holder,Holder
1,Milton,2018,2018-10-28,NC,1,R16,1,MITCHELL_Ethan,NZL,27.706849,17.0,2.0,10.225,70.415648,Holder,Holder,Holder,Holder,Holder,Holder
2,Milton,2018,2018-10-28,NC,1,R16,2,MORENO_SANCHEZ_Jose,ESP,24.958904,28.0,2.0,10.676,67.440989,Holder,Holder,Holder,Holder,Holder,Holder
3,Milton,2018,2018-10-28,NC,1,R16,3,RUDYK_Mateusz,POL,23.290411,9.0,1.0,10.204,70.560564,Holder,Holder,Holder,Holder,Holder,Holder
4,Milton,2018,2018-10-28,NC,1,R16,3,PERALTA_Juan,ESP,28.468493,27.0,2.0,10.293,69.950452,Holder,Holder,Holder,Holder,Holder,Holder
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1846,Milton,2023,2023-04-23,NC,2,F,G,PAUL_Nicholas,TTO,24.597260,1.0,1.0,10.143,70.984916,Holder,Holder,Holder,Holder,Holder,Holder
1847,Milton,2023,2023-04-23,NC,2,F,G,RUDYK_Mateusz,POL,27.778082,2.0,2.0,10.211,70.512193,Holder,Holder,Holder,Holder,Holder,Holder
1848,Milton,2023,2023-04-23,NC,2,F,B,RICHARDSON_Matthew,AUS,24.032877,3.0,1.0,11.393,63.196700,Holder,Holder,Holder,Holder,Holder,Holder
1849,Milton,2023,2023-04-23,NC,2,F,B,SAHROM_Muhammad_Shah_Firdaus,MAS,27.424658,4.0,2.0,16.795,42.869902,Holder,Holder,Holder,Holder,Holder,Holder


In [145]:
#Function for adding unique riders to array "rider"

def check_if_in_list(name):
    exists = name in rider
    if exists == False:
        a = (name)
        rider.append(str(a))
        
#implementing function

rider=[]
for i in range(len(df)):
    check_if_in_list(df.Athlete[i])
rider[0:5]



#Defining the Rider class which assigns the riders a name (their own) and a default skill

class Rider:
    def __init__(self, name, skill):
        self.skill = skill
        self.name = name
    
    def __lt__(self, other):
        return self.score < other.score
    
#Implementing function

for i in range(len(rider)):
    rider[i] = Rider(str(rider[i]), str(Rating()))
print(rider[5].skill)

#Defining the race function

skills=[]

def race(start):
    global skills
    j = start
    db = []
    starter = []
    rs = []
    order = []
    indexes = []
    rank = []
    word = "["
    while df.Rank[j] <= df.Rank[j+1]:
        index = next((k for k, item in enumerate(rider) if item.name == df.Athlete[j]), -1)
        indexes.append(index)
        rs.append('r'+str(j)) #maybe f strings?
        rank.append(int(df.Rank[j])-1)
        word = word + "(" + str(rider[index].skill).strip('trueskill.') + ",), "
        j+=1
    index = next((k for k, item in enumerate(rider) if item.name == df.Athlete[j]), -1)
    indexes.append(index)
    rs.append('r'+str(j))
    rank.append(int(df.Rank[j])-1)
    translation = {39: None}
    new = str(rs).translate(translation)
    word = word + "(" +str(rider[index].skill).strip('trueskill.') + ",)], ranks = " + str(rank) + ")"
    code_string = str(new) + " = rate(" + word
    exec(code_string)
    #print(rider[indexes[j]].skill+ "old")
    print(df.Location[j] + ' ' + df.Event[j] + ' ' + str(df.Stage[j]))
    for i in range(len(indexes)):
        x = str(eval(rs[i])).strip('(,)') + ')'
        rider[indexes[i]].skill=x
        skills.append(rider[indexes[i]].skill)
        
    start = j
    return start+1
    
    
#Looping through all races - The value counts thing is the number of first place finishes (not quite perfect)
#Just add a "stopper" race at the end
races = 0
start=0
events = int(df['Rank'].value_counts()[1])
for i in range(events+1):
    start = race(start)
    races +=1
    


trueskill.Rating(mu=25.000, sigma=8.333)
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R8
Milton NC R8
Milton NC R8
Milton NC R8
Milton NC R8
Milton NC R8
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC SF
Milton NC SF
Milton NC F
Milton NC F
Milton NC F
Milton NC F
Berlin NC R16
Berlin NC R16
Berlin NC R16
Berlin NC R16
Berlin NC R16
Berlin NC R16
Berlin NC R16
Berlin NC R16
Berlin NC R16
Berlin NC R16
Berlin NC R16
Berlin NC R8
Berlin NC R8
Berlin NC R8
Berlin NC R8
Berlin NC R8
Berlin NC R8
Berlin NC R8
Berlin NC QF
Berlin NC QF
Berlin NC QF
Berlin NC QF
Berlin NC QF
Berlin NC QF
Berlin NC SF
Berlin NC SF
Berlin NC SF
Berlin NC SF
Berlin NC F
Berlin NC F
London NC R16
London NC R16
London NC R16
London NC R16
London NC R16
London NC R16
London NC R16
London NC R16
London NC R16
London NC R8
London NC R8
London NC R8
London NC R8
London NC R8


Cairo NC QF
Cairo NC QF
Cairo NC QF
Cairo NC QF
Cairo NC QF
Cairo NC QF
Cairo NC SF
Cairo NC SF
Cairo NC SF
Cairo NC SF
Cairo NC SF
Cairo NC F
Cairo NC F
Cairo NC F
Cairo NC F
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R16
Milton NC R8
Milton NC R8
Milton NC R8
Milton NC R8
Milton NC R8
Milton NC R8
Milton NC R8
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC QF
Milton NC SF
Milton NC SF
Milton NC SF
Milton NC SF
Milton NC F
Milton NC F
Milton NC F
Milton NC F


KeyError: 1851

In [146]:
df

,Location,Year,Date,Event,Round,Stage,Heat,Athlete,Country,Age,Final_Rank,Rank,Time,Avg Speed,Initial_CSE,Final_CSE,Mu,Sigma,ExpectedRank,RatingChange
0,Milton,2018,2018-10-28,NC,1,R16,1,BARRETTE_Hugo,CAN,27.336986,16.0,1.0,10.178,70.740814,Holder,Holder,Holder,Holder,Holder,Holder
1,Milton,2018,2018-10-28,NC,1,R16,1,MITCHELL_Ethan,NZL,27.706849,17.0,2.0,10.225,70.415648,Holder,Holder,Holder,Holder,Holder,Holder
2,Milton,2018,2018-10-28,NC,1,R16,2,MORENO_SANCHEZ_Jose,ESP,24.958904,28.0,2.0,10.676,67.440989,Holder,Holder,Holder,Holder,Holder,Holder
3,Milton,2018,2018-10-28,NC,1,R16,3,RUDYK_Mateusz,POL,23.290411,9.0,1.0,10.204,70.560564,Holder,Holder,Holder,Holder,Holder,Holder
4,Milton,2018,2018-10-28,NC,1,R16,3,PERALTA_Juan,ESP,28.468493,27.0,2.0,10.293,69.950452,Holder,Holder,Holder,Holder,Holder,Holder
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1846,Milton,2023,2023-04-23,NC,2,F,G,PAUL_Nicholas,TTO,24.597260,1.0,1.0,10.143,70.984916,Holder,Holder,Holder,Holder,Holder,Holder
1847,Milton,2023,2023-04-23,NC,2,F,G,RUDYK_Mateusz,POL,27.778082,2.0,2.0,10.211,70.512193,Holder,Holder,Holder,Holder,Holder,Holder
1848,Milton,2023,2023-04-23,NC,2,F,B,RICHARDSON_Matthew,AUS,24.032877,3.0,1.0,11.393,63.196700,Holder,Holder,Holder,Holder,Holder,Holder
1849,Milton,2023,2023-04-23,NC,2,F,B,SAHROM_Muhammad_Shah_Firdaus,MAS,27.424658,4.0,2.0,16.795,42.869902,Holder,Holder,Holder,Holder,Holder,Holder


In [147]:
#giving everyone a skill estimate of mu-2*sig AFTER their race

#df=df.append( df.iloc[[-1]*3] )
for i in range(len(skills)):
    skill_split = skills[i].split(',')
    mu = float(skill_split[0][20:])
    sigma = float(skill_split[1].split(')')[0][7:])
    xx = mu - 2*sigma
    df['Final_CSE'][i] = xx
    df['Mu'][i] = mu
    df['Sigma'][i] = sigma
    
#Initial Rating. "Spots" is a list of all indexes where that athlete appears. 

for i in range(len(df)):
    name = df['Athlete'][i]
    spots = df.index[df['Athlete'] == name].tolist()
    #print(name)
    #print(spots)
    if i == spots[0]:
        df.Initial_CSE[i] = 9.00 #assigns baseline skill if this is there first appearance 
    else:
        spot_index = spots.index(i)
        df.Initial_CSE[i] = df.Final_CSE[spots[spot_index-1]] #assigns most recent "final_CSE" if they've appeared before
    #df.RatingChange[i] = df.Final_CSE[i] - df.Initial_CSE[i]
    
#This is for the expected rank. It creates a df for each heat, sorts it, then creates the "expected" array which is the 
#expected ranks. It may go [1,2,3,3,3,6] for example

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None
#warnings.filterwarnings(action='once')
i=0
while i <= len(df):
    df_temp = pd.DataFrame(columns = ["Athlete", "Initial_CSE","Index"])
    while df.Rank[i]<df.Rank[i+1]:
        df_temp=df_temp.append({'Athlete' : df.Athlete[i], 'Initial_CSE' : df.Initial_CSE[i], 'Index' : i}, ignore_index=True)
        i+=1
    df_temp=df_temp.append({'Athlete' : df.Athlete[i], 'Initial_CSE' : df.Initial_CSE[i], 'Index' : i}, ignore_index=True)
    i+=1
    df_temp = df_temp.sort_values(by = "Initial_CSE", ascending=False)
    df_temp = df_temp.reset_index(drop=True)
    expected =[1]
    for j in range(1,len(df_temp)):
        if df_temp.Initial_CSE[j] == df_temp.Initial_CSE[j-1]:
            #print("same" + str(df_temp.Initial_CSE[j]))
            expectation = expected[j-1]
            expected.append(expectation)
        else:
            expectation = len(expected) +1 
            expected.append(expectation)
    #print(df_temp)
    #print(expected)
    for k in range(len(expected)):
        index = df_temp.Index[k]
        df.ExpectedRank[index] = expected[k]


IndexError: list index out of range

In [149]:
df.drop(df.tail(1).index,inplace=True)
df = df.replace({'_': ' '}, regex=True)
df.to_csv(r'Sprint_Trueskill_Men.csv',index=False,encoding="utf-32")

In [148]:
df

,Location,Year,Date,Event,Round,Stage,Heat,Athlete,Country,Age,Final_Rank,Rank,Time,Avg Speed,Initial_CSE,Final_CSE,Mu,Sigma,ExpectedRank,RatingChange
0,Milton,2018,2018-10-28,NC,1,R16,1,BARRETTE_Hugo,CAN,27.336986,16.0,1.0,10.178,70.740814,9.0,16.639,30.109,6.735,Holder,Holder
1,Milton,2018,2018-10-28,NC,1,R16,1,MITCHELL_Ethan,NZL,27.706849,17.0,2.0,10.225,70.415648,9.0,10.499,22.443,5.972,Holder,Holder
2,Milton,2018,2018-10-28,NC,1,R16,2,MORENO_SANCHEZ_Jose,ESP,24.958904,28.0,2.0,10.676,67.440989,9.0,10.5,22.448,5.974,Holder,Holder
3,Milton,2018,2018-10-28,NC,1,R16,3,RUDYK_Mateusz,POL,23.290411,9.0,1.0,10.204,70.560564,9.0,15.054,29.396,7.171,Holder,Holder
4,Milton,2018,2018-10-28,NC,1,R16,3,PERALTA_Juan,ESP,28.468493,27.0,2.0,10.293,69.950452,9.0,6.262,20.604,7.171,Holder,Holder
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1846,Milton,2023,2023-04-23,NC,2,F,G,PAUL_Nicholas,TTO,24.597260,1.0,1.0,10.143,70.984916,30.394,30.53,32.51,0.99,Holder,Holder
1847,Milton,2023,2023-04-23,NC,2,F,G,RUDYK_Mateusz,POL,27.778082,2.0,2.0,10.211,70.512193,29.62,29.528,31.238,0.855,Holder,Holder
1848,Milton,2023,2023-04-23,NC,2,F,B,RICHARDSON_Matthew,AUS,24.032877,3.0,1.0,11.393,63.196700,30.25,30.35,32.872,1.261,Holder,Holder
1849,Milton,2023,2023-04-23,NC,2,F,B,SAHROM_Muhammad_Shah_Firdaus,MAS,27.424658,4.0,2.0,16.795,42.869902,23.549,23.468,26.308,1.42,Holder,Holder


In [11]:
1%4

1